In [ ]:
#Import
# pyproj removed - WGS84 polygon now built via geopandas.to_crs() in config
from shapely.geometry import box as shapely_box
import os, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


import osmnx as ox
import spaghetti

from libpysal import weights
from esda.moran import Moran


from scipy.spatial import cKDTree, Voronoi, voronoi_plot_2d
from scipy.spatial.distance import cdist


warnings.filterwarnings('ignore')
ox.settings.log_console = False

# CONFIGURATION

In [ ]:
# Centre of Leeds city centre (Briggate / The Headrow junction)
LEEDS_LAT  =  53.7997
LEEDS_LON  = -1.5492

# For accident CSV filtering (BNG metres)
BNG_EAST_MIN  = 430000   # western edge  (Easting)
BNG_EAST_MAX  = 431000   # eastern edge  (Easting)
BNG_NORTH_MIN = 433000   # southern edge (Northing)
BNG_NORTH_MAX = 434000   # northern edge (Northing)


ACCIDENT_DIR = "C:/Users/azaan/Desktop/Kings/SEM_2/Network_data_analysis/Coursework/accident_data"   # folder containing downloaded CSV files
CRS_METRIC   = "EPSG:27700"        # British National Grid — metres

# Approximate area in km² (used for density metrics in Task A)
AREA_KM2 = (
    (BNG_EAST_MAX  - BNG_EAST_MIN)  / 1000 *
    (BNG_NORTH_MAX - BNG_NORTH_MIN) / 1000
)

print(f"  Area: {AREA_KM2:.2f} km2")
# Build the study polygon in WGS84 for osmnx.graph_from_polygon().
# Alternative to pyproj: shapely_box makes a BNG rectangle (EPSG:27700),
# geopandas.to_crs() reprojects it to WGS84 (EPSG:4326) without pyproj.
_study_bbox_bng  = shapely_box(BNG_EAST_MIN, BNG_NORTH_MIN, BNG_EAST_MAX, BNG_NORTH_MAX)
STUDY_POLY_WGS84 = (gpd.GeoDataFrame(geometry=[_study_bbox_bng], crs=CRS_METRIC)
                    .to_crs("EPSG:4326").geometry.iloc[0])


# TASK A — SPATIAL NETWORKS & PLANARITY

In [ ]:
def task_a():
    print("\n")
    print("  TASK A — SPATIAL NETWORKS & PLANARITY")


    # Download road network from OpenStreetMap
    # network_type='drive' filters to roads used for driving only,
    # excluding footpaths, cycleways, and private roads.
    # graph_from_polygon() replaces graph_from_bbox() — no pyproj needed.
    # STUDY_POLY_WGS84 is built in the config cell via geopandas.to_crs().
    # This directly addresses the coursework requirement.
    print("\n  Fetching OSM drive network...")
    bounds = STUDY_POLY_WGS84.bounds   # (minLon, minLat, maxLon, maxLat)
    print(f"  Using WGS84 bounds: W={bounds[0]:.5f} S={bounds[1]:.5f} "
          f"E={bounds[2]:.5f} N={bounds[3]:.5f}")

    graph = ox.graph_from_polygon(
        STUDY_POLY_WGS84,
        network_type='drive',      # driving roads only — no footpaths/private
        simplify=True,             # merge degree-2 nodes into single edges
        retain_all=False           # keep only the strongly connected component --> largest component and removes tiny disconnected components
    )
    print(f"  Raw graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges") #nodes = intersections, Edges = road segments

    # Project to British National Grid (metric CRS)
    # Required for meaningful distance/area measurements in metres.
    graph_proj = ox.project_graph(graph, to_crs=CRS_METRIC) #Converts coordinates to metres using British National Grid (EPSG:27700)

    #Create 2 datasets
        # node-id, x-coordinate, y coordinate
        # u (road start), v(road end), length (metres), geometry (road shape)
    nodes_proj, edges_proj = ox.graph_to_gdfs(graph_proj)

    # Q1 — Coordinates of chosen area
    print(f"\n  Q1 — Chosen area bounding box:")
    print(f"       West:  {bounds[0]:.5f}E   East:  {bounds[2]:.5f}E")
    print(f"       South: {bounds[1]:.5f}N   North: {bounds[3]:.5f}N")
    print(f"       Centre: ({LEEDS_LAT}°N, {LEEDS_LON}°E) — Leeds Briggate")

    print(f" Approx area: {AREA_KM2:.3f} km²")

    # Q2 — Network characteristics
    # osmnx basic stats
    stats_basic = ox.basic_stats(graph_proj)

    # Street length stats
    edge_lengths = edges_proj['length'].values
    avg_street_len = edge_lengths.mean() #Average length of road segments

    # Node density = nodes per km²
    node_density = graph_proj.number_of_nodes() / AREA_KM2

    # Intersection density = intersections (degree ≥ 3) per km²
    # These represent actual junctions --> measures urban connectivity
    node_degrees  = dict(graph_proj.degree())
    intersections = sum(1 for d in node_degrees.values() if d >= 3)
    intersection_density = intersections / AREA_KM2

    # Edge density = total edge length (km) per km²
    total_edge_len_km = edge_lengths.sum() / 1000
    edge_density = total_edge_len_km / AREA_KM2

    # Spatial diameter = max shortest path in metres --> longest shortest distance between intersections
    # (computed on undirected projected graph)
    graph_und = ox.convert.to_undirected(graph_proj)
    # Sample diameter via eccentricity on largest component
    largest_cc = max(nx.connected_components(graph_und), key=len)
    graph_sub = graph_und.subgraph(largest_cc)
    # BFS-based approximation: find max shortest path from a sample of nodes
    sample_nodes = list(graph_sub.nodes())[:20]
    max_dist = 0
    for src in sample_nodes:
        lengths = nx.single_source_dijkstra_path_length(
            graph_sub, src, weight='length')
        max_dist = max(max_dist, max(lengths.values()))
    spatial_diameter = max_dist

    print(f"\n  Q2 — Network characteristics:")
    print(f"       Nodes:               {graph_proj.number_of_nodes()}")
    print(f"       Edges:               {graph_proj.number_of_edges()}")
    print(f"       Spatial diameter:    {spatial_diameter:.0f} m")
    print(f"       Avg street length:   {avg_street_len:.1f} m")
    print(f"       Node density:        {node_density:.1f} nodes/km²")
    print(f"       Intersection density:{intersection_density:.1f} intersections/km²")
    print(f"       Edge density:        {edge_density:.2f} km/km²")

    # Q3 — Average circuitry
    # Circuitry = actual network distance / Euclidean distance (≥ 1.0).
    # Measures how much extra travel roads impose vs a straight line.
    # Computed over a sample of node pairs for efficiency.

    sample      = list(graph.nodes())[:20]         # 20 source nodes
    node_pos    = {n: (nodes_proj.loc[n,'x'], nodes_proj.loc[n,'y'])
                   for n in graph_sub.nodes()
                   if n in nodes_proj.index}

    circuitry_vals = []
    for u in sample:
        # One BFS from u gives distances to ALL other nodes — reuse them all
        sp_lengths = nx.single_source_dijkstra_path_length(graph_sub, u, weight='length')
        ux, uy = node_pos.get(u, (None, None))
        if ux is None:
            continue
        for v, net_dist in sp_lengths.items():
            if v == u or v not in node_pos:
                continue
            vx, vy   = node_pos[v]
            euc_dist = np.sqrt((ux-vx)**2 + (uy-vy)**2)
            if euc_dist > 0:
                circuitry_vals.append(net_dist / euc_dist)

    avg_circuitry = np.mean(circuitry_vals)
    print(f"\n  Q3 — Average circuitry: {avg_circuitry:.4f}")
    print(f"       Interpretation: travellers drive {avg_circuitry:.2f}× further")
    print(f"       than the straight-line distance on average.")
    if avg_circuitry < 1.3:
        print(f"       → Highly efficient grid-like street layout.")
    elif avg_circuitry < 1.6:
        print(f"       → Moderately efficient — some indirect routing.")
    else:
        print(f"       → Low efficiency — many forced detours.")

    # Q4 — Planarity
    # A planar graph can be drawn with no crossing edges.
    # Conditions: Euler's formula V - E + F = 2; for simple planar: E ≤ 3V - 6.
    # Real road networks are ALMOST planar but not strictly so because:
    #   - Bridges and underpasses create grade-separated crossings
    #     (edges cross geometrically but don't share a node).
    #   - OSM topology may include "crossing" edges at motorway junctions.
    V = graph_und.number_of_nodes()
    E = graph_und.number_of_edges()
    planar_edge_limit = 3 * V - 6

    print(f"\n  Q4 — Planarity analysis:")
    print(f"       Nodes V = {V},  Edges E = {E}")
    print(f"       Planar limit (3V-6) = {planar_edge_limit}")
    print(f"       E ≤ 3V-6? -> {E <= planar_edge_limit}  "
          f"({'necessary condition MET' if E <= planar_edge_limit else 'VIOLATED'})")

    # networkx planarity check (exact, linear time)
    is_planar, certificate = nx.check_planarity(graph_und)
    print(f"       nx.check_planarity() -> {'PLANAR' if is_planar else 'NOT PLANAR'}")

    if not is_planar:
        print(f"       Non-planar because: bridges/flyovers create geometric")
        print(f"       crossings without shared intersection nodes in OSM.")
        print(f"       Example: a motorway flyover crosses a surface road —")
        print(f"       both are edges in the graph but share no node.")
    else:
        print(f"       Planar: the chosen 1 km² area contains no grade-separated")
        print(f"       crossings — all intersections are at-grade junctions.")

    # Road network map
    fig, ax = plt.subplots(figsize=(10, 10))
    ox.plot_graph(graph_proj, ax=ax, node_size=15, node_color='#E53935',
                  edge_color='#1565C0', edge_linewidth=1.5,
                  bgcolor='white', show=False, close=False)
    ax.set_title(
        f"Task A — Leeds Drive Network  ({BNG_EAST_MIN}-{BNG_EAST_MAX}E, "
        f"{BNG_NORTH_MIN}-{BNG_NORTH_MAX}N)\n"
        f"Nodes={V}  Edges={E}  AvgStreetLen={avg_street_len:.0f}m  "
        f"Circuitry={avg_circuitry:.3f}",
        fontsize=11, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig("task_a_road_network.png", dpi=130, bbox_inches='tight')

    return graph_proj, nodes_proj, edges_proj, AREA_KM2

# TASK B — ROAD ACCIDENTS

In [ ]:
 # Load 2014-2016 Leeds accident CSVs from ACCIDENT_DIR.
# These files store location as British National Grid:
    #'Grid Ref: Easting'  -> BNG Easting  (metres, x-axis)
    #'Grid Ref: Northing' -> BNG Northing (metres, y-axis)
    # 'Casualty Severity'  -> 'Slight' / 'Serious' / 'Fatal'

#Rows are filtered to BNG_EAST/NORTH_MIN/MAX defined in config.
 # 2017-2019 files (no coordinates) are automatically skipped.
 # Returns GeoDataFrame in EPSG:27700 — same CRS as road network.

def load_accidents():

    if not os.path.isdir(ACCIDENT_DIR):
        raise FileNotFoundError(
            f"Folder '{ACCIDENT_DIR}' not found.\n"
        )

    frames = []
    for fname in sorted(os.listdir(ACCIDENT_DIR)):
        if not fname.endswith('.csv'):
            continue

        df = pd.read_csv(os.path.join(ACCIDENT_DIR, fname), low_memory=False)
        df.columns = [c.strip() for c in df.columns]   # remove stray whitespace

        # Check file has location columns — 2017-2019 do not
        has_e = any('easting'  in c.lower() for c in df.columns)
        has_n = any('northing' in c.lower() for c in df.columns)
        if not (has_e and has_n):
            print(f"  [SKIP] {fname} — no Easting/Northing columns "
                  f"(columns: {list(df.columns)[:4]} ...)")
            continue

        # Standardise column names regardless of capitalisation/spacing
        col_map = {}
        for c in df.columns:
            cl = c.lower()
            if 'easting'  in cl: col_map[c] = 'easting'
            if 'northing' in cl: col_map[c] = 'northing'
            if 'date'     in cl: col_map[c] = 'date'
            if 'severity' in cl: col_map[c] = 'severity'
        df.rename(columns=col_map, inplace=True)

        # Convert coordinates to numeric (handles stray dashes/spaces)
        df['easting']  = pd.to_numeric(df['easting'],  errors='coerce')
        df['northing'] = pd.to_numeric(df['northing'], errors='coerce')
        df.dropna(subset=['easting', 'northing'], inplace=True)

        # Filter to study area bounding box (BNG metres)
        mask = (
            (df['easting']  >= BNG_EAST_MIN)  &
            (df['easting']  <= BNG_EAST_MAX)  &
            (df['northing'] >= BNG_NORTH_MIN) &
            (df['northing'] <= BNG_NORTH_MAX)
        )
        print(f"  {fname}: {len(df):,} rows -> {mask.sum()} in study area")
        frames.append(df[mask].copy())

    if not frames:
        raise ValueError(
            "No usable accident data found.\n"
            "Ensure 2014.csv, 2015.csv, 2016.csv are in ./accident_data/\n"
            "and BNG_EAST/NORTH bounds overlap your data."
        )

    df_all = pd.concat(frames, ignore_index=True) #concatenates the list of dataframes in order and returns new data frame
    print(f"\n  Total accidents in study area (all years): {len(df_all):,}")
    if len(df_all) < 300: # check accidents are more than 300
        print(f"  WARNING: {len(df_all)} accidents — requirement is 300+.")
        print(f"  Widen BNG_EAST/NORTH bounds by +/-500 m and re-run.")
    else:
        print(f"  300+ requirement met.")

    # Map severity text -> numeric code: Fatal=1, Serious=2, Slight=3
    sev_map = {'fatal': 1, 'serious': 2, 'slight': 3}
    if 'severity' in df_all.columns:
        df_all['severity_code'] = (
            df_all['severity'].str.strip().str.lower()
            .map(sev_map).fillna(3).astype(int)
        )
    else:
        df_all['severity_code'] = 3

    # Build GeoDataFrame — Easting/Northing are already EPSG:27700
    # No CRS conversion needed; same system as road network
    gdf = gpd.GeoDataFrame(
        df_all,
        geometry=gpd.points_from_xy(df_all['easting'], df_all['northing']),
        crs=CRS_METRIC
    )
    return gdf


def task_b(graph_proj, nodes_proj, edges_proj):
    print("\n")
    print("  TASK B - ROAD ACCIDENTS")

    gdf_acc = load_accidents()
    print(f"\n  Accident CRS: {gdf_acc.crs}")
    print(f"  Road network CRS: {CRS_METRIC}")
    print(f"  CRS match: {str(gdf_acc.crs) == CRS_METRIC or 'EPSG:27700' in str(gdf_acc.crs)}")

    # B1: Accident distribution map
    #Draw road network, overlay accidents and color by severity
    fig, ax = plt.subplots(figsize=(11, 10))
    edges_proj.plot(ax=ax, color='#90A4AE', linewidth=0.8, zorder=1)
    nodes_proj.plot(ax=ax, color='#B0BEC5', markersize=3,  zorder=2)

    severity_colors = {1: '#B71C1C', 2: '#F57F17', 3: '#1B5E20'}
    severity_labels = {1: 'Fatal', 2: 'Serious', 3: 'Slight'}
    for sev, color in severity_colors.items():
        subset = gdf_acc[gdf_acc['severity_code'] == sev]
        if len(subset):
            subset.plot(ax=ax, color=color, markersize=18,
                        alpha=0.75, zorder=3, label=severity_labels[sev])

    patches = [mpatches.Patch(color=c, label=severity_labels[s])
               for s, c in severity_colors.items()]
    ax.legend(handles=patches, title='Severity', fontsize=10, loc='upper right')
    ax.set_title(
        f"Task B1 — Road Accident Distribution  (n={len(gdf_acc):,}, 2014-2016)\n"
        f"BNG {BNG_EAST_MIN}-{BNG_EAST_MAX}E, {BNG_NORTH_MIN}-{BNG_NORTH_MAX}N",
        fontsize=11, fontweight='bold'
    )
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig("task_b1_accident_distribution.png", dpi=130, bbox_inches='tight')
    print("\n  Saved -> task_b1_accident_distribution.png")

    # B2: Snap accidents to nearest edge
    acc_coords     = np.array([[pt.x, pt.y] for pt in gdf_acc.geometry]) #numpy array of accident coordinates
    edge_centroids = np.array([[g.centroid.x, g.centroid.y]
                                for g in edges_proj.geometry]) #represents each edge by the x,y of its centroid
    edge_tree = cKDTree(edge_centroids) #Each accident to the nearest road edge
    _, eidx   = edge_tree.query(acc_coords, k=1) #query the tree with all accident coordinates to get the nearest edge index for each accident

    edges_proj = edges_proj.copy()
    edges_proj['accident_count'] = 0
    counts     = np.zeros(len(edges_proj), dtype=int)
    np.add.at(counts, eidx, 1)
    edges_proj['accident_count'] = counts

    # B2: Moran's I
    # Queen contiguity: edges sharing a node are spatial neighbours
    w = weights.Queen.from_dataframe(edges_proj, silence_warnings=True) #create a spatial weights object from your roads GeoDataFrame using contiguity-style neighbors
    w.transform = 'r'   # row-standardise
    # each row of the weight matrix is divided by the row sum (so neighbors' weights sum to 1 for each feature). Row-standardisation is common when comparing regions with differing numbers of neighbors.

    # compute Moran's I using your accident counts and the weight matrix
    moran = Moran(edges_proj['accident_count'].values, w, permutations=99)
    # constructs an esda.moran.Moran object which computes Moran’s I statistic, expectation, variance, and various significance measures (including permutation p-value and normal approximation p-value).

    print(f"\n  B2 - Moran's I:")
    print(f"       I = {moran.I:.4f}   E[I] = {moran.EI:.4f}   p = {moran.p_sim:.4f}")
    if moran.p_sim < 0.05:
        if moran.I > moran.EI:
            print(f"       -> Significant POSITIVE spatial autocorrelation (p<0.05)")
            print(f"          High-accident roads cluster near other high-accident roads.")
        else:
            print(f"       -> Significant NEGATIVE autocorrelation (p<0.05)")
    else:
        print(f"       -> No significant spatial autocorrelation (p>=0.05)")

    # B2: Ripley's K / L-function
    # Compares observed accident clustering to complete spatial randomness (CSR)
    # L(r) > r  =>  clustering at that distance scale
    acc_xy     = np.array([[pt.x, pt.y] for pt in gdf_acc.geometry]) #Build accident coordinate array:
    n          = len(acc_xy)
    study_area = (BNG_EAST_MAX  - BNG_EAST_MIN) * (BNG_NORTH_MAX - BNG_NORTH_MIN)
    lambda_    = n / study_area   # intensity: accidents per m2

    rng        = np.random.default_rng(42)
    sample_idx = rng.choice(n, min(300, n), replace=False) #Sample a subset of events to speed pairwise distance computation
    # picks up to 300 random accident indices (without replacement) using a fixed seed for reproducibility. Sampling makes the computation cheaper while yielding an approximate K.
    dists      = cdist(acc_xy[sample_idx], acc_xy) #computes Euclidean distances between a sample of points (rows = sampled accidents) and all accidents.
    radii = np.linspace(50, 600, 30) # r values at which K(r) will be evaluated.

    # For each r the code:
    # counts all pairs (sampled → all) within distance r: (dists <= r).sum(),
    # subtracts len(sample_idx) to remove self‑pairs (each sampled point distance 0 to itself),
    # divides by m * λ where m = len(sample_idx) and λ is point intensity (points per unit area).
    # That yields an empirical Ripley K̂(r) estimate: K̂(r) ≈ (1 / (m λ)) Σ_i (# neighbors within r of sampled i, excluding self).

    K_obs = [(np.sum(dists <= r) - len(sample_idx)) / (len(sample_idx) * lambda_)
             for r in radii] #empirical K(r) for each radius r
    K_csr = np.pi * radii**2 #computes the theoretical K(r) under Complete Spatial Randomness (homogeneous Poisson): K_CSR(r) = π r^2
    L_obs = np.sqrt(np.array(K_obs) / np.pi) #computes the L-transform L(r) = sqrt(K(r)/π).
    L_csr = radii

    idx200 = np.searchsorted(radii, 200) #finds the index for radius ≈ 200 m, prints observed vs CSR K at that scale

    #K_obs > K_csr --> evidence of clustering at 200 m, otherwise --> evidence consistent with dispersion or CSR.

    print(f"\n  B2 - K-function at r=200m:")
    print(f"       K_obs={K_obs[idx200]:.1f}   K_csr={K_csr[idx200]:.1f}")
    print(f"       -> {'Clustering (K_obs > K_csr)' if K_obs[idx200] > K_csr[idx200] else 'Dispersed / random'}")

    # B3: Accident proximity to intersections
    # spaghetti snaps accidents to the road network then reports
    # how far along each edge the accident fell.
    # Fraction 0 = at intersection node,  1 = far end of edge
    print(f"\n  B3 - Accident proximity to intersections (spaghetti)...")
    ntw     = spaghetti.Network(in_data=edges_proj)
    ntw.snapobservations(gdf_acc, 'accidents', attribute=True)
    snapped = spaghetti.element_as_gdf(ntw, pp_name='accidents', snapped=True) #Snaps accident points onto road network

    # fraction = dist_to_node / edge_length
    fractions = []
    for _, row in snapped.iterrows(): #iterates snapped points
        try:
            edge_id = row.get('edge_id')
            dist    = row.get('dist_to_node') #gets and computes distance,if missing returns none
            if edge_id is not None and dist is not None:
                arc_len = ntw.edges.get(edge_id)
                if arc_len and arc_len > 0:
                    fractions.append(min(dist / arc_len, 1.0)) #Computes the fractional position along the edge: distance from the node divided by the edge length
        except Exception:
            pass


    fractions = np.array(fractions)
    mean_frac = np.mean(fractions) #prints mean/median
    print(f"       Mean fraction:   {mean_frac:.4f}")
    print(f"       Median fraction: {np.median(fractions):.4f}")
    if mean_frac < 0.35:
        print(f"       -> Accidents cluster NEAR intersections (turning conflicts)")
    elif mean_frac > 0.5:
        print(f"       -> Accidents tend to occur MID-ROAD (speed-related)")
    else:
        print(f"       -> Accidents distributed evenly along road segments")

    # Figures B2 + B3

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle("Task B — Road Accident Analysis", fontsize=13, fontweight='bold')

    # Panel 0: L-function
    ax = axes[0]
    ax.plot(radii, L_obs, 'r-',  lw=2,   label='L_obs (empirical)')
    ax.plot(radii, L_csr, 'b--', lw=1.5, label='L_csr (CSR baseline)')
    ax.fill_between(radii, L_csr, L_obs,
                    where=(np.array(L_obs) > L_csr),
                    alpha=0.2, color='red', label='Clustering region')
    ax.set_xlabel("Radius r (m)"); ax.set_ylabel("L(r)")
    ax.set_title("Ripley's K / L-function\n(L_obs > L_csr = clustering)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Panel 1: Moran's I scatter
    ax    = axes[1]
    y     = edges_proj['accident_count'].values.astype(float)
    y_std = (y - y.mean()) / (y.std() + 1e-9)
    lag   = w.full()[0] @ y_std
    ax.scatter(y_std, lag, alpha=0.4, s=10, color='steelblue')
    ax.axhline(0, color='grey', lw=0.8)
    ax.axvline(0, color='grey', lw=0.8)
    m, b = np.polyfit(y_std, lag, 1)
    xs   = np.linspace(y_std.min(), y_std.max(), 100)
    ax.plot(xs, m*xs + b, 'r-', lw=1.5, label=f"slope~I={moran.I:.3f}")
    ax.set_xlabel("Standardised accident count"); ax.set_ylabel("Spatial lag")
    ax.set_title(f"Moran's I Scatter\nI={moran.I:.4f}  p={moran.p_sim:.4f}")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Panel 2: Fraction histogram
    ax = axes[2]
    ax.hist(fractions, bins=30, color='#4CAF50', edgecolor='white', alpha=0.85)
    ax.axvline(mean_frac, color='red',    lw=2,   linestyle='--',
               label=f'Mean={mean_frac:.3f}')
    ax.axvline(0.5,       color='orange', lw=1.5, linestyle=':',
               label='Midpoint=0.5')
    ax.set_xlabel("Fraction of road length from nearest intersection")
    ax.set_ylabel("Count")
    ax.set_title("B3 — Accident Location Along Road\n0=intersection  1=far end")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("task_b23_analysis.png", dpi=130, bbox_inches='tight')
    print("  Saved -> task_b23_analysis.png")

    return gdf_acc, edges_proj

TASK C — VORONOI DIAGRAMS & MARATHON ROUTE PLANNING


# Select N=4 seed points using two combined criteria:
# 1. Maximum spread — greedy farthest-point algorithm
# 2. Low accident risk — weighted by inverse accident density within 100 m
# Both criteria are in BNG metres, consistent with all other data.
# Returns list of (easting, northing) tuples and node IDs.

In [ ]:
def select_seed_points(graph_proj, nodes_proj, edges_proj, gdf_acc, N=4):

    acc_xy   = np.array([[pt.x, pt.y] for pt in gdf_acc.geometry]) #accident locations
    node_xy  = np.array([[r.geometry.x, r.geometry.y]
                          for _, r in nodes_proj.iterrows()]) #road network nodes (possible seeds)
    node_ids = list(nodes_proj.index) #node identifiers

    # Accident count within 100 m of each node
    # Build tree ONCE
    acc_tree  = cKDTree(acc_xy)
    acc_count = np.array([
        len(acc_tree.query_ball_point(xy, r=100))  # ← reused for all nodes
        for xy in node_xy
    ])
    # Invert: low accident count -> high score (preferred seed location)
    # normalise values between 0 and 1
    # high value --> dangerous area
    # low value --> safe area
    acc_norm = 1 - acc_count / (acc_count.max() + 1e-9)

    # Greedy farthest-point: start from centroid, iteratively pick the
    # node with the highest combined score (distance * safety)
    bounds   = nodes_proj.total_bounds   # Returns a length-4 sequence: [minx, miny, maxx, maxy]
    cx       = (bounds[0] + bounds[2]) / 2 #Compute the midpoint of the bounding box on the x- and y-axes respectively.
    cy       = (bounds[1] + bounds[3]) / 2
    seeds_xy = [(cx, cy)]
    seed_ids = []

    for _ in range(N):
        seeds_arr = np.array(seeds_xy)
        # cdist computes all node-to-seed distances in one vectorised call
        # min over seeds axis --> distance of each node to its nearest seed
        all_dists = cdist(node_xy, seeds_arr)
        min_dists = all_dists.min(axis=1)
        score     = min_dists * acc_norm
        best      = np.argmax(score)
        seeds_xy.append(tuple(node_xy[best]))
        seed_ids.append(node_ids[best])

    seeds_xy = seeds_xy[1:]   # remove initial centroid placeholder

    print(f"\n  C1 — Seed selection (farthest-point + low accident risk):")
    for i, (x, y) in enumerate(seeds_xy):
        print(f"    Seed {i+1}: Easting={x:.0f}m  Northing={y:.0f}m  "
              f"(safety score={acc_norm[list(node_xy).index(node_xy[node_ids.index(seed_ids[i])])]:.3f})"
              if seed_ids[i] in node_ids else f"    Seed {i+1}: ({x:.0f}, {y:.0f})")

    return seeds_xy, seed_ids # co-ordinates for voronoi diagram
                              # Node IDs for network analysis



# Node-network Voronoi diagram.


# Assign each node and edge in the road network to the nearest seed point, thus creating Voronoi cells on the network
def build_voronoi(seeds_xy, edges_proj, nodes_proj):
    seed_arr   = np.array(seeds_xy) # coordinates of selected seed points
    node_xy    = np.array([[r.geometry.x, r.geometry.y]
                            for _, r in nodes_proj.iterrows()]) #coordinates of every road network node
    node_ids   = list(nodes_proj.index) #node identifiers

    _, cell_id = cKDTree(seed_arr).query(node_xy, k=1) # build a KD-tree from seed points
    node_cell  = dict(zip(node_ids, cell_id)) #node cell mapping

    # Assign edges to cells
    # assign edge to same cell as its starting node
    # if something fails, assign -1 (unassigned)
    edge_cell  = [
        node_cell.get(row.name[0], -1) if hasattr(row.name, '__len__') else -1
        for _, row in edges_proj.iterrows()
    ]
    edges_out        = edges_proj.copy()
    edges_out['cell']= edge_cell  #now every edge has cell = which voronoi region it belongs to
    return node_cell, edges_out



# Find a circular route (~42 km) within a Voronoi cell.
# Strategy — greedy path extension:
# At each step, move to the neighbour whose edge length keeps
    # cumulative distance closest to the target (penalise revisits).
#Once within tolerance of target, close the loop via shortest
    # path back to start. Tries 5 different start nodes.
    # All distances are in metres (BNG projected graph).
    # Returns (route_node_list, length_km) or (None, 0).


def find_marathon_route(graph_proj, nodes_proj, cell_nodes,
                        target_km=42.0, tolerance_km=2.0):

    # convert units
    target_m    = target_km    * 1000
    tolerance_m = tolerance_km * 1000

    #restrict graph to one voronoi cell
    graph_sub = graph_proj.subgraph(cell_nodes).copy()
    if graph_sub.number_of_nodes() < 5:
        return None, 0

    graph_und      = ox.convert.to_undirected(graph_sub)
    largest_cc = max(nx.connected_components(graph_und), key=len) #keeps largest component
    if len(largest_cc) < 5:
        return None, 0
    graph_und      = graph_und.subgraph(largest_cc).copy()
    nodes_list = list(graph_und.nodes())
    best_route, best_len = None, 0

    for attempt in range(5): #try 5 different starting points
        start         = nodes_list[attempt % len(nodes_list)] #different starting nodes will give more chances to find a valid loop
        path          = [start]
        total         = 0.0
        visited_edges = set() #avoid repetition
        current       = start

        # Main loop - grow the route
        for _ in range(5000): # prevents infinite loop
            neighbours = list(graph_und.neighbors(current)) #possible next roads
            if not neighbours:
                break

            np.random.shuffle(neighbours) #avoids same path every time
            remaining = target_m - total #smart next node selection - GREEDY
            best_next, best_next_len, best_score = None, 0, float('inf')

            for nbr in neighbours: #Evaluate each neighbour
                edge_len = graph_und[current][nbr].get('length', 50)
                penalty  = 100 if (current, nbr) in visited_edges else 0
                score    = abs(remaining - edge_len) + penalty
                if score < best_score:
                    best_score    = score
                    best_next     = nbr
                    best_next_len = edge_len

            if best_next is None:
                break

            visited_edges.add((current, best_next)) #update the route
            visited_edges.add((best_next, current))
            path.append(best_next)
            total  += best_next_len #extend the path
            current = best_next

            # Once close enough, try to close the loop
            if total >= target_m - tolerance_m:
                try:
                    rp       = nx.shortest_path(graph_und, current, start, weight='length') #try shortest path back to start - creates a loop
                    rl       = sum(graph_und[rp[i]][rp[i+1]].get('length', 50)
                                   for i in range(len(rp) - 1))
                    full_len = total + rl #compute full route length
                    if abs(full_len - target_m) <= tolerance_m: #check if valid marathon length and within tolerance
                        full_route = path + rp[1:]
                        if (best_route is None or
                                abs(full_len - target_m) < abs(best_len - target_m)):
                            best_route, best_len = full_route, full_len
                        break
                except nx.NetworkXNoPath:
                    pass

            if total > target_m + tolerance_m:
                break   # overshot this attempt

    return best_route, best_len / 1000   # return km


def task_c(graph_proj, nodes_proj, edges_proj, gdf_acc, N=4):
    print("\n")
    print("  TASK C - VORONOI DIAGRAMS & MARATHON ROUTE PLANNING")


    # C1: Seed point selection
    seeds_xy, seed_ids = select_seed_points(
        graph_proj, nodes_proj, edges_proj, gdf_acc, N=N)

    # C2: Node-network Voronoi
    print("\n  C2 - Building node-network Voronoi diagram...")
    node_cell, edges_with_cell = build_voronoi(seeds_xy, edges_proj, nodes_proj)

    # Build Euclidean Voronoi for background visualisation
    # (padded with dummy points to close diagram at edges)
    seed_arr    = np.array(seeds_xy)
    bounds      = nodes_proj.total_bounds
    pad         = max(bounds[2]-bounds[0], bounds[3]-bounds[1]) * 2
    dummy       = np.array([
        [bounds[0]-pad, bounds[1]-pad], [bounds[2]+pad, bounds[1]-pad],
        [bounds[0]-pad, bounds[3]+pad], [bounds[2]+pad, bounds[3]+pad]
    ])
    vor         = Voronoi(np.vstack([seed_arr, dummy]))
    cell_colors = ['#BBDEFB', '#FFE0B2', '#C8E6C9', '#F3E5F5']
    markers     = ['*', 'D', '^', 's']

    fig, ax = plt.subplots(figsize=(11, 10))
    voronoi_plot_2d(vor, ax=ax, show_vertices=False, line_colors='navy',
                    line_width=1.5, line_alpha=0.6, point_size=0)

    for cell_idx, color in enumerate(cell_colors):
        sub = edges_with_cell[edges_with_cell['cell'] == cell_idx]
        if len(sub):
            sub.plot(ax=ax, color=color, linewidth=2.5, alpha=0.8, zorder=2)

    for i, (x, y) in enumerate(seeds_xy):
        ax.plot(x, y, markers[i], color='red', markersize=16,
                markeredgecolor='black', zorder=5, label=f'Seed {i+1}')

    gdf_acc.plot(ax=ax, color='black', markersize=4, alpha=0.3,
                 zorder=3, label='Accidents')
    ax.legend(fontsize=9, loc='upper right')
    ax.set_title(
        f"Task C2 — Node-Network Voronoi  "
        f"({BNG_EAST_MIN}-{BNG_EAST_MAX}E, {BNG_NORTH_MIN}-{BNG_NORTH_MAX}N)\n"
        f"Colours=cells  markers=seeds  dots=accidents",
        fontsize=11, fontweight='bold'
    )
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig("task_c2_voronoi.png", dpi=130, bbox_inches='tight')

    print("\n  Why node-network Voronoi?")
    print("    Euclidean Voronoi ignores roads; boundaries may cut through buildings.")
    print("    Node-network assigns each road junction to the nearest seed by")
    print("    BNG distance, giving meaningful road-reachable catchment areas.")
    print("    Edge-planar is more precise but computationally expensive.")

    # C3 & C4: Find ~42 km circular routes
    print("\n  C3/C4 - Searching for ~42 km circular routes per cell...")
    cells = {}
    for node, cell in node_cell.items():
        cells.setdefault(cell, []).append(node)

    routes_found = {}
    fig2, axes2  = plt.subplots(2, 2, figsize=(16, 14))
    fig2.suptitle(
        "Task C3/C4 — Marathon Route Planning (~42 km circuits)\n"
        "Node-network Voronoi cells with found routes",
        fontsize=13, fontweight='bold'
    )

    for cell_idx, ax in enumerate(axes2.flat):
        cell_nodes = cells.get(cell_idx, [])
        print(f"\n    Cell {cell_idx+1}: {len(cell_nodes)} nodes", end="  ")

        route, length_km = find_marathon_route(
            graph_proj, nodes_proj, cell_nodes,
            target_km=42.0, tolerance_km=2.0
        )

        # Draw cell background
        sub = edges_with_cell[edges_with_cell['cell'] == cell_idx]
        sub.plot(ax=ax, color=cell_colors[cell_idx], linewidth=1.5, alpha=0.7)

        if route and len(route) > 2:
            routes_found[cell_idx] = (route, length_km)
            print(f"Route found: {length_km:.2f} km")
            coords = [
                (nodes_proj.loc[n, 'geometry'].x,
                 nodes_proj.loc[n, 'geometry'].y)
                for n in route if n in nodes_proj.index
            ]
            if len(coords) > 1:
                rx, ry = zip(*coords)
                ax.plot(rx, ry, 'r-', lw=2.5, zorder=4,
                        label=f'Route ({length_km:.1f} km)')
                ax.plot(rx[0], ry[0], 'go', markersize=12,
                        zorder=5, label='Start/End')
        else:
            print(f"No valid route found")
            ax.text(0.5, 0.5,
                    f'Cell {cell_idx+1}\nNo 42 km circuit\n({len(cell_nodes)} nodes)',
                    transform=ax.transAxes, ha='center', va='center',
                    fontsize=11, color='#B71C1C')

        sx, sy = seeds_xy[cell_idx]
        ax.plot(sx, sy, markers[cell_idx], color='blue', markersize=14,
                markeredgecolor='black', zorder=6, label=f'Seed {cell_idx+1}')
        ax.legend(fontsize=8, loc='upper right')
        ax.set_title(f"Cell {cell_idx+1} — "
                     f"{'Route found' if cell_idx in routes_found else 'No route'}")
        ax.set_axis_off()

    plt.tight_layout()
    plt.savefig("task_c34_routes.png", dpi=130, bbox_inches='tight')


    #  C5: Handle cells with no valid route
    no_route = [i for i in range(N) if i not in routes_found]
    print(f"\n  C5 - Cells without a route: {[i+1 for i in no_route]}")
    print("""
  Options considered:
    A - More seeds (N>4): smaller cells, less road length each
    B - Expand study area: more road km per cell  <- chosen
    C - Relax single-cell constraint: allow brief cross-boundary excursions
    D - Edge-planar Voronoi: finer boundaries for more road length per cell

  Chosen option B: double the study area bounding box
  """)

    # Expand BNG bounds by 50% in each direction
    expand       = 0.5
    east_range   = BNG_EAST_MAX  - BNG_EAST_MIN
    north_range  = BNG_NORTH_MAX - BNG_NORTH_MIN
    E2_MIN = BNG_EAST_MIN  - east_range  * expand
    E2_MAX = BNG_EAST_MAX  + east_range  * expand
    N2_MIN = BNG_NORTH_MIN - north_range * expand
    N2_MAX = BNG_NORTH_MAX + north_range * expand

    # Build expanded WGS84 polygon via geopandas (replaces pyproj _transformer).
    _exp_bbox_bng   = shapely_box(E2_MIN, N2_MIN, E2_MAX, N2_MAX)
    _exp_poly_wgs84 = (gpd.GeoDataFrame(geometry=[_exp_bbox_bng], crs=CRS_METRIC)
                       .to_crs("EPSG:4326").geometry.iloc[0])

    print(f"  Expanded area: {E2_MIN:.0f}-{E2_MAX:.0f}E, "
          f"{N2_MIN:.0f}-{N2_MAX:.0f}N  (BNG metres)")

    try:
        G2      = ox.graph_from_polygon(
            _exp_poly_wgs84,
            network_type='drive', simplify=True, retain_all=False
        )
        G2_proj = ox.project_graph(G2, to_crs=CRS_METRIC)
        n2, e2  = ox.graph_to_gdfs(G2_proj)
        s2, _   = select_seed_points(G2_proj, n2, e2, gdf_acc, N=N)
        c2, e2c = build_voronoi(s2, e2, n2)
        cells2  = {}
        for node, cell in c2.items():
            cells2.setdefault(cell, []).append(node)

        routes2 = {}
        for ci in range(N):
            r2, l2 = find_marathon_route(
                G2_proj, n2, cells2.get(ci, []),
                target_km=42.0, tolerance_km=2.5
            )
            if r2:
                routes2[ci] = (r2, l2)
                print(f"    Cell {ci+1}: route {l2:.2f} km")
            else:
                print(f"    Cell {ci+1}: no route")

        print(f"\n  After expansion: {len(routes2)}/{N} cells have a valid route "
              f"(was {len(routes_found)}/{N})")
    except Exception as e:
        print(f"  Expansion step skipped: {e}")

    return routes_found

# Main Code

In [ ]:
if __name__ == "__main__":
    np.random.seed(42)

    graph_proj, nodes_proj, edges_proj, AREA_KM2 = task_a()  # task_a returns 4 values
    gdf_acc, edges_with_accidents        = task_b(graph_proj, nodes_proj, edges_proj)
    routes                               = task_c(graph_proj, nodes_proj,
                                                  edges_with_accidents,
                                                  gdf_acc, N=4)

    print("\n\nAll Part 2 tasks complete.")
    print("Output files:")
    print("  task_a_road_network.png")
    print("  task_b1_accident_distribution.png")
    print("  task_b23_analysis.png")
    print("  task_c2_voronoi.png")
    print("  task_c34_routes.png")
